# Walkthrough 2: from mini-batch to prediction, then the full training loop (Base-RelBench)

Continues [Walkthrough 1](01_data_graph_sampling.ipynb). Everything here calls the
real code in [`experiments/train.py`](../experiments/train.py), so what you see is what
the baseline runs. Steps 2–8 take **one small mini-batch** (4 seeds, fan-out `[4, 2]`)
through the model and recompute three of them by hand; Steps 9–10 run the **full
training loop** used for the baseline.

| Step | What you see |
|---|---|
| 1 | Setup: graph, task, one mini-batch, the model |
| 2 | Column encoders: one value encoded by hand |
| 3 | Row encoder: one 128-d vector per row |
| 4 | Relative-time encoding → `h⁽⁰⁾` |
| 5 | One GraphSAGE layer, recomputed by hand |
| 6 | Second layer and head → prediction |
| 7 | Loss, recomputed by hand |
| 8 | One optimiser step |
| 9 | Full loop on `driver-top3`: epochs, best-epoch selection, test score |
| 10 | Full loop on `driver-position`: clipping and MAE |

If you re-run the notebook, exact numbers may differ slightly from the saved outputs
(neighbour sampling is random); every hand-vs-model check still matches.

## Setup

**Does:** imports the libraries and `train.py` (our training script).  
**In:** nothing.  
**Out:** nothing printed.

In [1]:
import os, sys, warnings
from collections import Counter
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'walkthroughs' else Path.cwd()
os.environ.setdefault('HF_HOME', str(ROOT / 'data' / 'hf'))
sys.path.insert(0, str(ROOT / 'experiments'))
warnings.filterwarnings('ignore')

import numpy as np
import torch
import torch.nn.functional as F
import torch_frame
from relbench import load_dataset
from torch_geometric.seed import seed_everything

import train  # experiments/train.py

torch.set_printoptions(precision=4, sci_mode=False, linewidth=140)
cpu = torch.device('cpu')

## Step 1: setup

**Does:** builds the graph with `train.load_graph` (text columns skipped), the task's settings with `train.task_spec`, and a loader with tiny settings; draws one mini-batch and creates a fresh `BaseModel`.  
**In:** `rel-f1`, the `driver-top3` task, settings `batch_size=4`, `num_neighbors=4`.  
**Out:** the 4 seed drivers and their labels, the parameter count (4.15 M) and the model structure.

In [2]:
seed_everything(0)
dataset = load_dataset('rel-f1')
task = dataset.load_task('driver-top3')
data, col_stats = train.load_graph(dataset, task, 'none', cpu)
spec = train.task_spec(task)

small = train.parse_args(['--batch-size', '4', '--num-neighbors', '4'])  # fan-out [4, 2]
batch = next(iter(train.make_loaders(data, task, small)['train']))

model = train.BaseModel(data, col_stats, channels=128, out_channels=1, num_layers=2, aggr='sum')
print('seed drivers:', batch['drivers'].n_id.tolist(), '| labels:', batch['drivers'].y.tolist())
print('parameters:', sum(p.numel() for p in model.parameters()))
print(model)

Fetching 24 files:   0%|          | 0/24 [00:00<?, ?it/s]

Fetching 24 files: 100%|██████████| 24/24 [00:00<00:00, 1237.99it/s]

seed drivers: [17, 112, 30, 68] | labels: [0.0, 0.0, 1.0, 0.0]
parameters: 4150401
BaseModel(
  (encoder): HeteroEncoder(
    (encoders): ModuleDict(
      (races): ResNet(
        (encoder): StypeWiseFeatureEncoder(
          (encoder_dict): ModuleDict(
            (categorical): EmbeddingEncoder(
              (emb): Embedding(61, 128, padding_idx=0)
            )
            (numerical): LinearEncoder()
            (timestamp): TimestampEncoder(
              (positional_encoding): PositionalEncoding()
              (cyclic_encoding): CyclicEncoding()
            )
          )
        )
        (backbone): Sequential(
          (0): FCResidualBlock(
            (lin1): Linear(in_features=512, out_features=128, bias=True)
            (lin2): Linear(in_features=128, out_features=128, bias=True)
            (relu): ReLU()
            (dropout): Dropout(p=0.2, inplace=False)
            (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
            (norm2): LayerN

## Step 2: column encoders

Each table has its own `torch_frame.nn.ResNet`. Its first part turns each
column into a 128-d vector according to the column's type. A numerical value is
standardised with the column's training statistics, then scaled and shifted by learned
vectors: $e_{v,c} = \frac{x_{v,c} - \mu_c}{\sigma_c}\,\mathbf w_c + \mathbf b_c$.

**Does:** encodes every column of the sampled `results` rows into a 128-d vector, then recomputes one numerical value by hand.  
**In:** the `results` rows of the batch: 11 columns (10 numerical + 1 date).  
**Out:** a `[rows, 11, 128]` tensor, and the same 128 numbers for `laps` computed by hand and by the model; ✓.

In [3]:
tf = batch['results'].tf
enc = model.encoder.encoders['results']        # torch_frame ResNet for `results`
x_cols, col_names = enc.encoder(tf)            # one vector per (row, column)
print('results rows in batch:', tf.num_rows, '| columns:', col_names)
print('per-column embeddings:', tuple(x_cols.shape), '= [rows, columns, channels]')

lin = enc.encoder.encoder_dict['numerical']    # LinearEncoder
j = tf.col_names_dict[torch_frame.numerical].index('laps')
x = tf.feat_dict[torch_frame.numerical][0, j]
by_hand = (x - lin.mean[j]) / lin.std[j] * lin.weight[j] + lin.bias[j]
print(f"\nlaps = {x.item():.0f}, column mean {lin.mean[j]:.1f}, std {lin.std[j]:.1f}")
print('first 5 channels by hand :', by_hand[:5].detach())
print('first 5 channels by model:', x_cols[0, col_names.index('laps'), :5].detach())
assert torch.allclose(by_hand, x_cols[0, col_names.index('laps')], atol=1e-5)
print('match ✓')

results rows in batch: 12 | columns: ['fastestLap', 'grid', 'laps', 'milliseconds', 'number', 'points', 'position', 'positionOrder', 'rank', 'statusId', 'date']
per-column embeddings: (12, 11, 128) = [rows, columns, channels]

laps = 71, column mean 44.1, std 32.0
first 5 channels by hand : tensor([ 0.0113, -0.0074, -0.0006,  0.0021,  0.0077])
first 5 channels by model: tensor([ 0.0113, -0.0074, -0.0006,  0.0021,  0.0077])
match ✓


## Step 3: row encoder

`model.eval()` switches dropout off so that the checks below are exact.

**Does:** runs the full encoder of every table: the column vectors of a row are concatenated and passed through 4 residual blocks and a decoder.  
**In:** the batch's `TensorFrame`s, one per table.  
**Out:** one 128-d vector per sampled row, for every table (empty tables give 0 rows).

In [4]:
model.eval()
with torch.no_grad():
    x_dict = model.encoder(batch.tf_dict)
for nt, x in x_dict.items():
    print(f'{nt:22s} {batch[nt].tf.num_cols:2d} columns → {tuple(x.shape)}')

races                   4 columns → (33, 128)
qualifying              3 columns → (12, 128)
constructor_standings   4 columns → (0, 128)
standings               4 columns → (12, 128)
constructors            1 columns → (7, 128)
drivers                 1 columns → (4, 128)
constructor_results     2 columns → (0, 128)
circuits                3 columns → (0, 128)
results                11 columns → (12, 128)


## Step 4: relative-time encoding → $h^{(0)}$

**Does:** for every timed row, computes how many days before its seed's prediction time it happened, encodes that number with sinusoids and a linear layer, and adds it to the row vector. Static tables get nothing.  
**In:** row vectors from Step 3, row times, seed times.  
**Out:** `h0`: same shapes as Step 3; checks that `h0 = row vector + time encoding` for `results` and that `drivers` is unchanged.

In [5]:
seed_time = batch['drivers'].seed_time
days = (seed_time[batch['results'].batch] - batch['results'].time) / 86400
print('results rows: days before their seed time:', days[:6].tolist())

with torch.no_grad():
    rel_time = model.temporal_encoder(seed_time, batch.time_dict, batch.batch_dict)
    h0 = model.encode(batch, task.entity_table)   # encoder + time, exactly as in training
print('time encoding added to:', sorted(rel_time))
assert torch.allclose(h0['results'], x_dict['results'] + rel_time['results'])
assert torch.equal(h0['drivers'], x_dict['drivers'])
print('h0[results] = row embedding + time encoding ✓ ; drivers unchanged ✓')

results rows: days before their seed time: [1134.0, 1050.0, 777.0, 315.0, 222.0, 306.0]
time encoding added to: ['constructor_results', 'constructor_standings', 'qualifying', 'races', 'results', 'standings']
h0[results] = row embedding + time encoding ✓ ; drivers unchanged ✓


## Step 5: one GraphSAGE layer, by hand

For relation $r$, the message into node $v$ is
$m_{v,r} = W_{r,1} \sum_{u \in \mathcal N_r(v)} h_u + W_{r,2}\, h_v + b_r$.
`HeteroConv` sums the messages of all relations ending at $v$'s type; then a per-type
LayerNorm and a ReLU: $h^{(1)}_v = \mathrm{ReLU}(\mathrm{LN}(\sum_r m_{v,r}))$.

**Does:** recomputes the first GraphSAGE layer for seed driver 0 with plain tensor operations, and compares it with PyG's `HeteroConv` output.  
**In:** `h0` and the batch edges that end at `drivers`.  
**Out:** the neighbours per relation, and the first 6 numbers of the new driver vector by hand and by the model; ✓.

In [6]:
conv, norms = model.gnn.convs[0], model.gnn.norms[0]
v = 0
total = torch.zeros(128)
with torch.no_grad():
    for (src, rel, dst), ei in batch.edge_index_dict.items():
        if dst != 'drivers':
            continue
        sage = conv.convs[(src, rel, dst)]
        nbrs = ei[0, ei[1] == v]                       # neighbours of v under this relation
        total += sage.lin_l(h0[src][nbrs].sum(0)) + sage.lin_r(h0['drivers'][v])
        print(f'{rel:22s} from {src:12s}: {len(nbrs)} neighbours')
    h1_by_hand = F.relu(norms['drivers'](total.unsqueeze(0)))[0]
    h1_model = {k: F.relu(norms[k](x)) for k, x in conv(h0, batch.edge_index_dict).items()}
print('\nby hand :', h1_by_hand[:6])
print('by model:', h1_model['drivers'][v, :6])
assert torch.allclose(h1_by_hand, h1_model['drivers'][v], atol=1e-5)
print('match ✓')

f2p_driverId           from qualifying  : 4 neighbours
f2p_driverId           from standings   : 4 neighbours
f2p_driverId           from results     : 4 neighbours

by hand : tensor([0.0000, 0.7966, 1.7478, 0.0000, 0.0000, 0.0000])
by model: tensor([0.0000, 0.7966, 1.7478, 0.0000, 0.0000, 0.0000])
match ✓


A relation adds $W_{r,2} h_v + b_r$ even when $v$ has no neighbours under it, so a seed
with an empty subgraph (Walkthrough 1, Step 8) still gets an update, built from its own
features only.

## Step 6: second layer and head → prediction

**Does:** runs the whole model: layer 2 brings in information from 2 hops away; the head (one linear layer) maps each seed driver's final vector to one number, the *logit*.  
**In:** the mini-batch.  
**Out:** 4 logits, their probabilities (`sigmoid`), and the true labels.

In [7]:
with torch.no_grad():
    logits = model(batch, task.entity_table).view(-1)
print('logits       :', logits)
print('probabilities:', logits.sigmoid())
print('labels       :', batch['drivers'].y)

logits       : tensor([0.3338, 0.5586, 0.3179, 0.3365])
probabilities: tensor([0.5827, 0.6361, 0.5788, 0.5833])
labels       : tensor([0., 0., 1., 0.], dtype=torch.float64)


## Step 7: loss, by hand

$\mathcal L = -\frac1B\sum_i \big[y_i \log\sigma(\hat y_i) +
(1-y_i)\log(1-\sigma(\hat y_i))\big]$

**Does:** computes binary cross-entropy by hand and with the task's loss function (`BCEWithLogitsLoss`).  
**In:** the 4 logits and labels.  
**Out:** both values; ✓.

In [8]:
y = batch['drivers'].y.float()
p = logits.sigmoid()
by_hand = -(y * p.log() + (1 - y) * (1 - p).log()).mean()
by_torch = spec.loss_fn(logits, y)
print(f'by hand {by_hand:.6f} | BCEWithLogitsLoss {by_torch:.6f}')
assert torch.allclose(by_hand, by_torch, atol=1e-6)
print('match ✓')

by hand 0.826770 | BCEWithLogitsLoss 0.826770
match ✓


## Step 8: one optimiser step

**Does:** back-propagates the loss and lets Adam (lr 0.005) update every parameter once, then recomputes the loss on the same batch.  
**In:** the model and the mini-batch.  
**Out:** the largest gradients, which parameters got no gradient, and the loss before and after the step.

In [9]:
def batch_loss():  # loss on this batch with dropout off, so before/after are comparable
    model.eval()
    with torch.no_grad():
        return spec.loss_fn(model(batch, task.entity_table).view(-1), y).item()

loss_before = batch_loss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
model.train()   # dropout on, as in real training
loss = spec.loss_fn(model(batch, task.entity_table).view(-1), y)
optimizer.zero_grad()
loss.backward()
grads = {n: p.grad.norm().item() for n, p in model.named_parameters() if p.grad is not None}
optimizer.step()

print('largest gradient norms:')
for n, g in sorted(grads.items(), key=lambda kv: -kv[1])[:5]:
    print(f'  {g:8.4f}  {n}')
no_grad = [n for n, p in model.named_parameters() if p.grad is None]
def group(name):
    parts = name.split('.')
    return '.'.join(parts[:3]) if parts[0] in ('encoder', 'gnn') else '.'.join(parts[:2])
print(f'\n{len(no_grad)} parameter tensors got no gradient (they did not affect this loss):')
for g, n in sorted(Counter(map(group, no_grad)).items()):
    print(f'  {n:3d}  {g}')
print(f'\nloss on this batch: before {loss_before:.4f} → after one step {batch_loss():.4f}')

largest gradient norms:
    2.5671  encoder.encoders.drivers.encoder.encoder_dict.timestamp.weight
    1.2700  encoder.encoders.drivers.decoder.2.weight
    0.9218  encoder.encoders.drivers.backbone.0.lin2.weight
    0.8560  encoder.encoders.drivers.backbone.1.lin1.weight
    0.8509  encoder.encoders.drivers.backbone.0.lin1.bias

268 parameter tensors got no gradient (they did not affect this loss):
   40  encoder.encoders.circuits
   42  encoder.encoders.constructor_results
   42  encoder.encoders.constructor_standings
   45  gnn.convs.0
   69  gnn.convs.1
   10  gnn.norms.0
   16  gnn.norms.1
    4  temporal_encoder.lin_dict



loss on this batch: before 0.8268 → after one step 0.5216


Parameters get no gradient when they could not influence the loss: the encoders of the
three tables with no sampled rows, the first-layer convolutions of the relations that
touch those tables, and the **second-layer** convolutions and norms of every node type
except `drivers`. Only the seed drivers' final vectors feed the head, so layer-2
updates of other node types are computed but never used.

## Step 9: the full training loop on `driver-top3`

**Does:** runs `train.fit` at the real settings (512 seeds, `[128, 64]` neighbours). Each epoch trains on all training seeds, predicts the validation seeds and scores them; the epoch with the best validation ROC-AUC is kept (ties go to the later epoch) and scored once on the test split.  
**In:** a fresh model, loaders at the real settings, 3 epochs (the baseline uses 10).  
**Out:** one line per epoch (training loss, validation ROC-AUC), then the best epoch with its validation and test scores.

In [10]:
seed_everything(0)
args = train.parse_args(['--epochs', '3'])   # defaults: lr 0.005, batch 512, [128, 64]
model = train.BaseModel(data, col_stats, 128, spec.out_channels, 2, 'sum')
out = train.fit(model, train.make_loaders(data, task, args), task, spec, args, cpu)
print('\nkept epoch', out['best_epoch'], '| test', out['test'])

epoch 01  loss 0.6942  val roc_auc 0.6198


epoch 02  loss 0.4574  val roc_auc 0.6417


epoch 03  loss 0.4456  val roc_auc 0.6371


best epoch 2: val roc_auc 0.6417  test roc_auc 0.7312

kept epoch 2 | test {'roc_auc': 0.7312330163043479}


## Step 10: regression on `driver-position`

**Does:** repeats Step 9 for a regression task. Its spec uses the L1 loss, selects epochs on NMAE (MAE divided by the standard deviation of the training labels), also reports raw MAE, and clips every prediction to the 2nd–98th percentile range of the training labels.  
**In:** the `driver-position` task, a fresh model, 2 epochs.  
**Out:** the spec, the per-epoch scores, the test scores, and the smallest and largest validation predictions (both inside the clipping range).

In [11]:
reg_task = dataset.load_task('driver-position')
reg_spec = train.task_spec(reg_task)
print('loss:', type(reg_spec.loss_fn).__name__, '| tune on:', reg_spec.tune_metric,
      '| reported:', [m.__name__ for m in reg_spec.metrics], '| clip range:', reg_spec.clip)

seed_everything(0)
reg_data, reg_stats = train.load_graph(dataset, reg_task, 'none', cpu)
reg_loaders = train.make_loaders(reg_data, reg_task, train.parse_args(['--epochs', '2']))
model = train.BaseModel(reg_data, reg_stats, 128, reg_spec.out_channels, 2, 'sum')
out = train.fit(model, reg_loaders, reg_task, reg_spec, train.parse_args(['--epochs', '2']), cpu)

pred = train.predict(model, reg_loaders['val'], reg_spec, reg_task.entity_table, cpu)
print(f'\nvalidation predictions: min {pred.min():.2f}, max {pred.max():.2f} '
      f'(clip range {reg_spec.clip})')

loss: L1Loss | tune on: nmae | reported: ['nmae', 'mae'] | clip range: (2.0, 31.0)


epoch 01  loss 8.8814  val nmae 0.6047  val mae 4.2482


epoch 02  loss 5.8217  val nmae 0.6287  val mae 4.4169


best epoch 1: val nmae 0.6047  val mae 4.2482  test nmae 0.7040  test mae 4.9462



validation predictions: min 9.08, max 9.16 (clip range (2.0, 31.0))


After only 2 epochs the regression model predicts almost the same value (about 9) for every driver: with an L1 loss, a constant close to the median label is an easy early solution. No prediction reaches the clipping range here, so clipping changes nothing in this short run; `test_regression_predictions_are_clipped` in `experiments/tests/` checks the clipping itself by forcing extreme outputs. The baseline trains for 10 epochs.

## Recap

- `h⁽⁰⁾` = per-table ResNet over the column vectors, plus a time encoding for timed
  tables. This is the input the paper's filtering step (A) will gate.
- Each GraphSAGE layer sums one `SAGEConv` message per relation, then applies LayerNorm
  and ReLU. That sum over relations is where the paper's gated relations (step C) will
  plug in.
- `train.fit` is the whole baseline: 10 epochs, keep the best validation epoch, score it
  once on test. The baseline runs it for 3 tasks × 5 seeds.